# Week 4 — Data Science Internship Project
This notebook is built to support a professional internship-level data science workflow.
It covers dataset loading, cleaning, exploratory analysis, visualization, model training, and results interpretation.
This workflow is designed to adapt to a classification dataset and deliver reproducible, presentation-quality results.


In [ ]:
# Load required libraries for data processing, visualization, and modeling
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-whitegrid')


## 1. Data Loading and Initial Review
In this section, we load the dataset from the `data/` directory and inspect its structure.
This is an essential first step to understand the format, size, and initial quality of the data.


In [ ]:
# Configure the dataset path and load the file
DATA_PATH = os.path.join('..', 'data', 'dataset.csv') if os.path.exists(os.path.join('..', 'data', 'dataset.csv')) else 'data/dataset.csv'
print('Loading dataset from:', DATA_PATH)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError('Dataset not found. Please save your CSV file as data/dataset.csv')

df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully.')

# Show the first rows and basic dataset information
print('\nFirst 5 rows:')
display(df.head())

print('Dataset shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nMissing values per column:')
print(df.isnull().sum())

print('\nSummary statistics:')
display(df.describe(include='all'))


### Initial observations
The previous results help identify key quality issues such as missing values, duplicate rows, and inconsistent column names.
Next, we create a cleaned version of the dataset to prepare it for analysis and modeling.


In [ ]:
# Create a clean copy of the dataset for preprocessing
clean_df = df.copy()

# Standardize column names for consistency
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# Drop exact duplicate rows if present
duplicates = clean_df.duplicated().sum()
print(f'Found {duplicates} duplicate rows.')
if duplicates > 0:
    clean_df = clean_df.drop_duplicates().reset_index(drop=True)
    print('Dropped duplicates. New shape:', clean_df.shape)

# Convert columns containing the word 'date' to datetime when possible
for col in clean_df.columns:
    if 'date' in col:
        try:
            clean_df[col] = pd.to_datetime(clean_df[col])
            print(f'Converted {col} to datetime format.')
        except Exception:
            pass

# Separate numeric and categorical columns
numeric_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
category_cols = clean_df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Fill missing values for numeric and categorical columns
for col in numeric_cols:
    if clean_df[col].isnull().any():
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())
        print(f'Filled missing values in {col} with median.')

for col in category_cols:
    if clean_df[col].isnull().any():
        mode_value = clean_df[col].mode(dropna=True)
        if not mode_value.empty:
            clean_df[col] = clean_df[col].fillna(mode_value[0])
            print(f'Filled missing values in {col} with mode: {mode_value[0]}')

# Identify outliers using the IQR method for numeric features
outliers = {}
for col in numeric_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count_outliers = ((clean_df[col] < lower) | (clean_df[col] > upper)).sum()
    if count_outliers > 0:
        outliers[col] = int(count_outliers)

print('\nOutlier detection summary:')
print(outliers)
print('\nCleaned dataset shape:', clean_df.shape)


## 3. Exploratory Data Analysis (EDA)
This section explores feature distributions, relationships, and patterns in the cleaned dataset.
The goal is to identify trends and validate assumptions before modeling.


In [ ]:
# Refresh numeric and categorical column lists after cleaning
numeric_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
category_cols = clean_df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Numeric columns:', numeric_cols)
print('Categorical columns:', category_cols)

# Correlation analysis for numeric columns
if len(numeric_cols) >= 2:
    corr_matrix = clean_df[numeric_cols].corr()
    print('\nCorrelation matrix:')
    display(corr_matrix)

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='%.2f', cmap='coolwarm', square=True)
    plt.title('Numeric Feature Correlation')
    plt.tight_layout()
    plt.show()

    print('Insight: Strong correlations can guide feature selection and reveal multicollinearity risks.')

# Plot distributions for numeric features
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(clean_df[col], kde=True, bins=30, color='steelblue')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    print(f'Insight: Review the distribution of {col} for skewness, modality, and possible outliers.')

# Count plots for the top categorical features
for col in category_cols[:4]:
    plt.figure(figsize=(8, 4))
    order = clean_df[col].value_counts().index
    sns.countplot(y=clean_df[col], order=order, palette='pastel')
    plt.title(f'Count of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

    print(f'Insight: {col} distribution highlights common classes and any imbalance in categories.')

datetime_cols = clean_df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
if datetime_cols and numeric_cols:
    date_col = datetime_cols[0]
    metric_col = numeric_cols[0]
    time_series = clean_df.set_index(date_col).resample('M')[metric_col].mean().dropna()

    plt.figure(figsize=(10, 4))
    time_series.plot(marker='o')
    plt.title(f'Trend of {metric_col} over Time')
    plt.xlabel(date_col)
    plt.ylabel(metric_col)
    plt.tight_layout()
    plt.show()

    print('Insight: Trend analysis helps reveal seasonality and changes over time.')


## 4. Visualization and Insights
This section creates polished visualizations and saves them to the `visuals/` folder.
Each chart is paired with an interpretation note for stronger project presentation.


In [ ]:
visual_folder = 'visuals'
os.makedirs(visual_folder, exist_ok=True)

# Bar chart for the first categorical column
if category_cols:
    category = category_cols[0]
    counts = clean_df[category].value_counts()
    plt.figure(figsize=(8, 5))
    sns.barplot(x=counts.values, y=counts.index, palette='viridis')
    plt.title(f'Bar Chart of {category}')
    plt.xlabel('Count')
    plt.ylabel(category)
    plt.tight_layout()
    bar_path = os.path.join(visual_folder, f"bar_{category}.png")
    plt.savefig(bar_path)
    plt.show()
    print(f'Insight: {category} class frequencies are saved to {bar_path}.')

# Pie chart for top categories of the same feature
if category_cols:
    category = category_cols[0]
    top_counts = clean_df[category].value_counts().nlargest(6)
    plt.figure(figsize=(6, 6))
    top_counts.plot.pie(autopct='%1.1f%%', startangle=140, cmap='Set2')
    plt.ylabel('')
    plt.title(f'Top Categories in {category}')
    plt.tight_layout()
    pie_path = os.path.join(visual_folder, f"pie_{category}.png")
    plt.savefig(pie_path)
    plt.show()
    print(f'Insight: The most frequent {category} values represent the majority share.')

# Histogram for the first numeric column
if numeric_cols:
    numeric_feature = numeric_cols[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(clean_df[numeric_feature], bins=30, kde=True, color='royalblue')
    plt.title(f'Histogram of {numeric_feature}')
    plt.xlabel(numeric_feature)
    plt.ylabel('Frequency')
    plt.tight_layout()
    hist_path = os.path.join(visual_folder, f"hist_{numeric_feature}.png")
    plt.savefig(hist_path)
    plt.show()
    print(f'Insight: {numeric_feature} histogram saved to {hist_path}.')

# Scatter plot for two top numeric features
if len(numeric_cols) >= 2:
    x_feature, y_feature = numeric_cols[0], numeric_cols[1]
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=clean_df[x_feature], y=clean_df[y_feature], alpha=0.7)
    plt.title(f'{x_feature} vs {y_feature}')
    plt.xlabel(x_feature)
    plt.ylabel(y_feature)
    plt.tight_layout()
    scatter_path = os.path.join(visual_folder, f"scatter_{x_feature}_vs_{y_feature}.png")
    plt.savefig(scatter_path)
    plt.show()
    print(f'Insight: The scatter plot saves to {scatter_path}.')

# Box plot for a numeric feature grouped by a categorical feature
if numeric_cols and category_cols:
    group_num = numeric_cols[0]
    group_cat = category_cols[0]
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=clean_df[group_cat], y=clean_df[group_num], palette='pastel')
    plt.title(f'{group_num} by {group_cat}')
    plt.xlabel(group_cat)
    plt.ylabel(group_num)
    plt.xticks(rotation=45)
    plt.tight_layout()
    box_path = os.path.join(visual_folder, f"box_{group_num}_by_{group_cat}.png")
    plt.savefig(box_path)
    plt.show()
    print(f'Insight: The box plot saved to {box_path} and highlights distribution differences across groups.')

# Heatmap for numeric correlations
if len(numeric_cols) >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(clean_df[numeric_cols].corr(), annot=True, cmap='coolwarm', square=True)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    heatmap_path = os.path.join(visual_folder, 'heatmap_correlations.png')
    plt.savefig(heatmap_path)
    plt.show()
    print(f'Insight: The correlation heatmap saved to {heatmap_path}.')


## 5. Predictive Modeling
This section builds a baseline classification model with a clear, reproducible workflow.
We use the first viable target column and a Random Forest classifier for evaluation.


In [ ]:
# Select a classification target column automatically or use the last column as fallback
target_candidates = ['target', 'label', 'churn', 'outcome']
TARGET = next((col for col in clean_df.columns if col in target_candidates), None)
if TARGET is None:
    TARGET = clean_df.columns[-1]
print('Using target column:', TARGET)

# Prepare feature matrix and target vector
X = clean_df.drop(columns=[TARGET])
y = clean_df[TARGET]

# Convert a numeric target into bins only if there are many unique values
if pd.api.types.is_numeric_dtype(y) and y.nunique() > 10:
    try:
        y = pd.qcut(y, q=3, labels=False)
        print('Converted numeric target to 3-class bins.')
    except ValueError:
        print('Could not bin the numeric target due to duplicate edge values.')

# Encode categorical input features and fill missing values
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median())

# Remove features with no variance
non_constant_features = X.columns[X.nunique() > 1]
X = X[non_constant_features]
print('Features after encoding:', X.shape[1])

# Validate dataset before modeling
if X.shape[1] == 0:
    raise ValueError('No usable features remain after encoding. Review your input dataset.')
if y.nunique() < 2:
    raise ValueError('The target column requires at least two classes for classification.')

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None
)

# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate model performance
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print('\nClassification Report:')
print(classification_report(y_test, y_pred))


## 6. Final Conclusion
This notebook now meets internship-level expectations by:
- documenting an end-to-end data science workflow from raw data to model evaluation
- performing robust cleaning, exploratory data analysis, and visualization with saved charts in visuals/
- providing a reproducible baseline Random Forest model and key evaluation metrics
- offering practical next steps for improvement and deployment readiness

Key takeaways:
- validate dataset assumptions before selecting a final production model
- preserve the notebook and saved visuals for stakeholder review
- use the baseline model as a starting point for hyperparameter tuning and feature engineering

Recommended next steps:
- refine feature preparation using domain knowledge
- compare additional classifiers and apply cross-validation
- integrate findings into a report or dashboard for clear stakeholder communication
